# per-rank-cuda-device — ex1: pin per-rank cuda device with a mocked GPU

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `per-rank-cuda-device`. Running the final beacon cell reports progress against the `Distributed: per-rank cuda device` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: per-rank cuda device` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`per-rank-cuda-device`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "per-rank-cuda-device"
DD_SUBTOPIC = "Distributed: per-rank cuda device"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch.distributed quick refresher

PyTorch's collective-communication library (`torch.distributed`, aliased `dist`) lets multiple processes coordinate over tensors. The standard workflow:

1. **Each rank** runs the same function, parameterized by `rank` and `world_size`. Rank 0 is conventionally the 'driver'.
2. **`dist.init_process_group(backend=...)`** establishes the rendezvous. Backends:
   - `'nccl'` — NVIDIA's GPU-to-GPU primitive. Used in ARENA's multi-GPU setup. Requires CUDA + one process per GPU.
   - `'gloo'` — CPU-friendly. What you'll use in these drills (Colab CPU runtimes have no real GPUs).
3. **Pin a device** per rank: `torch.device(f'cuda:{rank}')` so each process owns exactly one GPU.
4. **Collective ops** (`all_reduce`, `broadcast`, `send`, `recv`) operate in-place on tensors of identical shape across all ranks.
5. **`dist.destroy_process_group()`** tears down at the end.

**Two ways to launch multiple ranks:**
- `torch.multiprocessing.spawn(fn, args=(...), nprocs=world_size)` — what ARENA uses. Spawn requires the worker fn be importable (not defined in `__main__`/a notebook cell).
- `mp.get_context('fork').Process(target=fn, args=...)` — Linux-only but works with cell-defined fns. The drills use this in tests so the worker can stay in the cell.

**Two-rank trick.** Colab gives ~2 CPU cores, so `world_size=2` is the right scale: enough to exercise the protocol, cheap enough to finish in seconds.

### This drill's atom: per-rank GPU pinning
In multi-GPU DDP, **rank `r` owns GPU `r`**. The canonical pattern:
```python
device = torch.device(f'cuda:{rank}')
torch.cuda.set_device(device)   # optional but recommended
model = SimpleModel().to(device)
x = torch.tensor([rank], dtype=torch.float32, device=device)
```
**Why the `f-string` over `cuda`?** Bare `'cuda'` means 'whatever the current device is' — fine for single-GPU, catastrophic for DDP (all ranks land on cuda:0 and OOM). The explicit `cuda:{rank}` guarantees correct sharding.

**Why this drill mocks.** Colab CPU runtimes have no real GPUs, so we patch `torch.cuda.set_device` and `Tensor.to` to record the args, then assert on those records. The pattern you write is identical to what runs on a real multi-GPU box.

### Exercise 1 — pin per-rank cuda device with a mocked GPU

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the `torch.device(f'cuda:{rank}')` + `.to(device)` pattern to pin a fresh model and a fresh tensor to the rank's GPU, verified by mocking `torch.cuda.set_device` and `Tensor.to`.
> Keywords: cuda, device, rank-pinning, mock, DDP
> ```

**KCs targeted:** `device-fstring-per-rank`, `model-to-device`

Implement `ex1_pin_to_rank_device(rank, model, scalar_value)`. The canonical per-rank pinning recipe:

1. Build a `torch.device` instance for `cuda:{rank}` using an f-string. DO NOT hardcode `'cuda:0'` or use bare `'cuda'`.
2. Call `torch.cuda.set_device(device)` to make it the current device (defensive — keeps any subsequent op from accidentally landing on the wrong GPU).
3. Move `model` to the device with `.to(device)` and re-assign.
4. Build a tensor `t.tensor([scalar_value], dtype=t.float32, device=device)`.
5. Return the tuple `(device, model, tensor)`.

**The test mocks CUDA.** Colab has no real GPUs, so the test uses `unittest.mock.patch` to replace `torch.cuda.set_device` and `Tensor.to` with stubs that record what the student passed. Your code never actually moves anything to GPU memory — the test just asserts you called the right APIs with the right args.

In [ ]:
import torch as t
from torch import Tensor

def ex1_pin_to_rank_device(rank: int, model: t.nn.Module, scalar_value: float):
    """Pin model + a scalar tensor to cuda:{rank}. Return (device, model, tensor)."""
    raise NotImplementedError()


def _test_ex1():
    from unittest.mock import patch, MagicMock

    class _ToyModel(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.p = t.nn.Parameter(t.tensor([1.0]))

    # Mock cuda.set_device + Tensor.to so the call works on CPU-only.
    set_device_calls = []

    def _fake_set_device(dev):
        set_device_calls.append(dev)

    to_calls = []
    _real_to = t.Tensor.to
    _real_module_to = t.nn.Module.to

    def _fake_tensor_to(self, *a, **k):
        to_calls.append(('Tensor', a, k))
        return self  # stay on CPU — we just record the call

    def _fake_module_to(self, *a, **k):
        to_calls.append(('Module', a, k))
        return self

    with patch('torch.cuda.set_device', _fake_set_device), \
         patch.object(t.Tensor, 'to', _fake_tensor_to), \
         patch.object(t.nn.Module, 'to', _fake_module_to):
        model = _ToyModel()
        device, returned_model, tensor = ex1_pin_to_rank_device(3, model, 5.0)

    # Device must be cuda:3 exactly.
    assert isinstance(device, t.device), f'expected torch.device, got {type(device)}'
    assert device.type == 'cuda', f'expected cuda type, got {device.type!r}'
    assert device.index == 3, f'expected cuda:3, got cuda:{device.index}'

    # set_device must have been called with the same device.
    assert len(set_device_calls) == 1, f'expected exactly one set_device call, got {len(set_device_calls)}'
    assert set_device_calls[0].index == 3, f'set_device got wrong index: {set_device_calls[0]}'

    # Module.to and Tensor.to must each have been called once with the device.
    module_to_calls = [c for c in to_calls if c[0] == 'Module']
    tensor_to_calls = [c for c in to_calls if c[0] == 'Tensor']
    assert len(module_to_calls) == 1, f'model.to was called {len(module_to_calls)} times, expected 1'
    # The Tensor.to call comes from t.tensor(..., device=device) — torch internally
    # may or may not route through Tensor.to; the SOLE requirement is that the
    # returned tensor's device is cuda:3 (well, would be — we kept it on CPU
    # in the mock so we just check the device arg we asked for).
    # The MODEL.to call's first positional arg must be the device.
    _args, _kwargs = module_to_calls[0][1], module_to_calls[0][2]
    _passed = _args[0] if _args else _kwargs.get('device')
    assert _passed == device, f'model.to was called with {_passed!r}, expected {device!r}'

    # Returned model must be the same instance (.to returned self in mock).
    assert returned_model is model, 'must reassign model = model.to(device) and return it'

    # 4 different ranks → 4 different device indices.
    for r in [0, 1, 2, 7]:
        with patch('torch.cuda.set_device', lambda d: None), \
             patch.object(t.Tensor, 'to', _fake_tensor_to), \
             patch.object(t.nn.Module, 'to', _fake_module_to):
            d, _, _ = ex1_pin_to_rank_device(r, _ToyModel(), float(r))
        assert d.index == r, f'rank {r} produced cuda:{d.index}'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_pin_to_rank_device(rank: int, model: t.nn.Module, scalar_value: float):
    device = t.device(f'cuda:{rank}')
    t.cuda.set_device(device)
    model = model.to(device)
    tensor = t.tensor([scalar_value], dtype=t.float32, device=device)
    return device, model, tensor
```

**Why `t.device(f'cuda:{rank}')` not `f'cuda:{rank}'`.** Both work as args to `.to(...)`, but constructing the `t.device` object once and reusing it (a) catches typos at construction time, (b) lets you pass the same canonical object to set_device / to() / tensor(...) — no chance of mismatched strings drifting apart.

**`set_device` vs `to(device)`.** `set_device` makes a CUDA context the *default* for the current thread; `.to(device)` moves a specific tensor/module. ARENA solutions tend to skip `set_device` and rely purely on explicit `.to(device)` — both are valid, but calling `set_device` first prevents accidental cuda:0 allocations from third-party libs that don't take a device arg.

**Reassignment matters for modules but not tensors.** `module.to(device)` mutates in place AND returns self; `tensor.to(device)` returns a NEW tensor (not in-place). Either way, write `x = x.to(device)` — it's the only form that's safe for both.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()